# 问题——大模型接口调用

**学习目标**

1. 学会在 Bigmodel 开放平台创建账号及了解开放平台能力
2. 掌握调用模型 API 进行问答的方法
3. 了解 temperature、top_p、max_tokens 参数对于模型输出结果的影响
4. 了解流式输出和非流式输出的不同，并掌握在不同场景使用这两种输出的方式

**作业说明**

1. 本作业需要同学补全少量代码，以使功能正常运行
2. 请在注释 #### 并标注下划线__________的地方补全相关代码。注释 # 的位置为普通注释，无需补全
3. 运行 Jupyter Notebook 全部代码块后，将文件导出为 HTML

## 准备工作

**了解大模型接口调用的基础知识**

在开始编程前，我们需要了解一些关于大语言模型的基本概念:

- Token: 模型处理文本的基本单位，通常一个英文单词是1-2个token，一个汉字大约是1个token
- API: 应用程序接口，允许我们与大模型进行交互
- 参数: 控制模型生成结果的各种设置

**安全管理**

在实际项目中，API密钥应该通过环境变量或配置文件管理，而不是硬编码在代码中。本教程为了简化演示，直接在代码中使用API密钥，但这不是最佳实践

**从 zhipuai 库中导入 ZhipuAI 类**

In [81]:
from zhipuai import ZhipuAI

## 任务1：调用大模型接口，查看对话结果

**我们先来看看开放平台的 API 都能给我们提供什么信息。定义一个名为 Message 的函数，用于向 API 发送消息并获取响应**

In [84]:
def Message(prompt):
    # 创建一个 ZhipuAI 的客户端实例，用于与开放平台的 API 交互
    client = ZhipuAI(
        #### 填写你的 api_key，查看 https://bigmodel.cn/usercenter/proj-mgmt/apikeys
        api_key="______________________",    
        # base_url 是智谱开放平台的基础 URL，指定了请求的目标地址
        base_url="https://open.bigmodel.cn/api/paas/v4/",
    )

    # 发起对话请求
    completion = client.chat.completions.create(
        model="______________________",   #### 补全模型名称，使用 glm-4-flash
        messages=[{"role": "user", "content":prompt}]
    )
    return completion

# 定义一个变量 user_prompt，存储用户要发送的消息
user_prompt="______________________"     #### 补全 prompt，比如“你是谁？”

# 查看返回结果
bot_ans=Message(user_prompt)
print(bot_ans)

Completion(model='glm-4-flash', created=1740448152, choices=[CompletionChoice(index=0, finish_reason='stop', message=CompletionMessage(content='我是一个名为 ChatGLM 的人工智能助手，是基于清华大学 KEG 实验室和智谱 AI 公司于 2024 年共同训练的语言模型开发的。我的任务是针对用户的问题和要求提供适当的答复和支持。', role='assistant', tool_calls=None))], request_id='20250225094911524bbd57f0b64d76', id='20250225094911524bbd57f0b64d76', usage=CompletionUsage(prompt_tokens=8, completion_tokens=47, total_tokens=55))


**可以看到，API 返回了很多我们暂时不需要的信息，请对 bot_ans 进行处理，仅保留模型回答内容**

In [86]:
bot_content=bot_ans.____________________    #### 补全 bot_ans 后面的代码，仅保留回答内容
print(bot_content)    

我是一个名为 ChatGLM 的人工智能助手，是基于清华大学 KEG 实验室和智谱 AI 公司于 2024 年共同训练的语言模型开发的。我的任务是针对用户的问题和要求提供适当的答复和支持。


**现在我们可以定义 Chat 函数，仅查看模型输出的对话内容**

In [88]:
# 为了方便后续的调用，我们将 api_key, base_url, 以及需要用到的模型存在变量中
api_key = ______________________     #### 填写你的 api_key，数据类型为字符串
base_url = "https://open.bigmodel.cn/api/paas/v4/"
model = ______________________    #### 补全模型名称，数据类型为字符串，使用 glm-4-flash

In [89]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model, 
        messages=[{"role": "user", "content":prompt}],
    )
    return ______________________    #### 补全返回结果，仅保留回答内容

user_prompt='你是谁？'
bot_ans=Chat(user_prompt)
print(bot_ans)

我是一个名为 ChatGLM 的人工智能助手，是基于清华大学 KEG 实验室和智谱 AI 公司于 2024 年共同训练的语言模型开发的。我的任务是针对用户的问题和要求提供适当的答复和支持。


## 任务2：探索 temperature、top_p、max_tokens 参数对于输出的影响

**在上一个任务中，我们已经学会了让大模型根据 prompt 生成回答。下面我们进一步探索如何通过不同参数来控制模型的生成内容**

### 2.1 temperature - 采样温度

Temperature参数控制输出的随机性和创造性:

- 取值范围: [0.0, 1.0]
- 值越小: 输出更确定和稳定
- 值越大: 输出更随机和创造性
- glm-4-flash 默认 temperature 值: 0.95

**先看看如何确保模型输出稳定的结果**

In [94]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model, 
        messages=[{"role": "user", "content":prompt}],
        temperature=______________________     #### 设置 temperature 值，使随机性最低
    )
    return completion.choices[0].message.content

user_prompt='写一段50字的笑话'
for i in range(5):
    bot_ans = Chat(user_prompt)
    print(bot_ans)

小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”


**再看看如何增强内容生成的创意性**

In [96]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        temperature=______________________     #### 设置 temperature 值，使随机性最强
    )
    return completion.choices[0].message.content

user_prompt='写一段50字的笑话'
for i in range(5):
    bot_ans = Chat(user_prompt)
    print(bot_ans)

小明的妈妈问他：“你喜欢苹果还是香蕉？”小明想了想说：“我喜欢‘苹果’和‘香蕉’一起吃。”妈妈问：“为什么？”小明笑眯眯地说：“因为这样我就可以吃‘苹果香蕉’啦！”
一个学生问老师：“为什么数学题里总要有字母？”老师回答：“因为这样我们就可以把你的成绩给隐藏起来。”同学们都笑了。
为什么猪总是开心？因为它们在“乐（肉）园”（乐圆）生活啊！😄
为什么猪不能成为钢琴家？因为它总是把五线谱踩在脚下！哈哈哈！
小明的妈妈问：“你今天吃了什么？”小明答：“我吃了一只鸡，一个蛋，还有三片面包。”妈妈惊讶：“这么多？那你是不是还喝了杯牛奶？”小明：“没有，我都吃了。”


**GLM-4 系列模型的 temperature 值默认为 0.95，更偏向于需要创造力的场景。如需要更稳定的输出结果，需要手动设置一个较低的 temperature 值**

### 2.2 top_p - 核采样概率阈值

Top_p是另一种控制随机性的方法，通过限制选择词汇的概率质量来工作:

- 取值范围: [0.0, 1.0]
- top_p=0.1: 只考虑概率质量最高的前10%的token
- glm-4-flash 默认 top_p 值: 0.7
- 建议只调整temperature或top_p其中之一

**同我们在 temperature 环节做的，先看看如何确保输出稳定的结果**

In [100]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        top_p=______________________     #### 设置 top_p 值，使随机性最低
    )
    return completion.choices[0].message.content

user_prompt='写一段50字的笑话'
for i in range(5):
    bot_ans = Chat(user_prompt)
    print(bot_ans)

小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”
小明问：“为什么猪圈里没有猪？”小华答：“因为猪都飞了！”小明疑惑：“猪会飞？”小华笑：“是啊，飞到天上去了！”


**再看看如何增强创意性**

In [102]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        top_p=______________________     #### 设置 top_p 值，使随机性最强
    )
    return completion.choices[0].message.content

user_prompt='写一段50字的笑话'
for i in range(5):
    bot_ans = Chat(user_prompt)
    print(bot_ans)

有一天，小明问妈妈：“妈妈，为什么别人都叫我‘小白兔’，我却叫我‘小黑狗’？”妈妈笑说：“因为别人都是‘白说’，而你总是‘黑说’！”小明哭笑不得。
小明问：“为什么大象不怕蛇？”小华答：“因为它们有牙！”小明疑惑：“那小猪呢？”小华笑说：“因为它们有猪牙！”
小明问：“为什么我跑步总是落后？”爸爸回答：“因为你总是‘小’步慢跑。”小明恍然大悟，原来是自己太矮了！
为什么电脑生病了还笑？因为它“中病毒”了！😂
小明问：“鱼儿为什么不敢上网？”老李答：“因为它怕网鱼！”哈哈，鱼儿上网，网鱼也上网，真是网络无限，鱼儿也疯狂！


### 2.3 max_tokens - 回复最大长度

Max_tokens控制模型生成的最大token数量:

- 不同模型支持的max_tokens范围不同
- glm-4-flash最大支持 4095 tokens
- 设置过小会导致回答被截断
- 设置过大可能导致不必要的资源消耗

**先来保证尽可能完整的输出**

In [105]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        max_tokens=______________________     #### 设置 max_tokens 值，让输出的内容尽可能完整，glm-4-flash 最大支持 4095
    )
    return completion.choices[0].message.content

user_prompt='怎样才能幸福'
bot_ans = Chat(user_prompt)
print(bot_ans)

幸福是一个主观的感受，不同的人有不同的理解和追求方式。以下是一些普遍认为有助于提升幸福感的方法：

1. **积极的心态**：保持乐观、积极的心态，对生活充满希望，有助于提升幸福感。

2. **健康的生活方式**：均衡饮食、适量运动、充足睡眠，这些都是保持身体健康的基础，也是幸福生活的重要组成部分。

3. **良好的人际关系**：与家人、朋友保持良好的关系，相互支持、理解和关爱，是幸福感的重要来源。

4. **有意义的工作**：找到自己热爱的工作，或者在工作中找到乐趣和成就感，能够显著提升幸福感。

5. **终身学习**：不断学习新知识、新技能，不仅可以提升自我价值感，也能增加生活的乐趣。

6. **感恩**：学会感恩，珍惜身边的人和事，能够让人更加满足和幸福。

7. **目标与规划**：为自己设定合理的目标，并为之努力，实现目标的过程本身就是一种幸福。

8. **自我实现**：追求自我价值的实现，找到自己的兴趣和激情所在，活出真实的自己。

9. **心理健康**：关注心理健康，学会应对压力和负面情绪，必要时寻求专业帮助。

10. **社会责任**：参与社会公益活动，帮助他人，也能获得内心的满足和幸福感。

每个人的情况不同，找到适合自己的幸福之道是关键。希望这些建议能对你有所帮助。


**再来理解 max_tokens 的控制效果：它是一种通过“截断”来控制生成长度的方式，而不是使模型恰好在这么多 tokens 内回答完**

In [107]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        max_tokens=______________________     #### 设置一个合适的 max_tokens 值，使模型的回答被截断
    )
    return completion.choices[0].message.content

user_prompt='怎样才能幸福'
bot_ans = Chat(user_prompt)
print(bot_ans)

幸福是一个主观的感受，不同的人有不同的幸福观。以下是一些普遍认为有助于提升幸福感的方法：

1. **积极的心态**：保持乐观


## 任务3：掌握流式输出和非流式输出

**输出方式对比**

- 非流式输出: 一次性返回完整结果，适合需要整体处理结果的场景
- 流式输出: 逐块返回结果，适合实时交互和长文本生成场景

**参数设置**

在 client.chat.completions.create 中：
- stream=False：非流式输出
- stream=True：流式输出

### 3.1 非流式输出

操作简单但是所有内容生成完才返回结果（长对话需要等很久）

**非流式输出适合的场景**
1. 需要对结果进行后处理的情况
2. 批量处理大量请求
3. 简短回答场景
4. 系统集成: 结果需要传递给其他系统组件

In [112]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,  
        base_url=base_url,
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content":prompt}],
        stream=______________________      #### 补全代码，选择非流式输出
    )
    return completion.choices[0].message.content


user_prompt="""
你现在同时存在于两个意识层面：

【创作者视角】 你是一位中国作家, 师从雷蒙德・卡佛

在寂静的午夜咖啡馆 凝视着窗外落下的小雨 正在写一个短篇小说 这个小说集成了你对老师的写作风格之大成 文字简洁凝练 用简短的句子和朴素的词汇表达深刻的情感和复杂的主题，以达到震撼人心的效果

刻意省略一些不必要的细节和修饰，让读者自己去填补文本中的空白，从而更加深入地参与到故事的解读中

【存在者视角】 面前的咖啡还在冒着热气, 钢笔在纸上划出的声音 像是生命在纸上最后的爬行, 你能闻到墨水、咖啡和雨水混合的气息, 那个杂揉味道，让你想起了...

---

让你的意识在这两个视角间 自由振荡 不需要刻意区分谁是作者、谁是角色 让它们在你的 consciousness 中 自然纠缠...

现在 让我看见你在写什么 让我感受那些文字是如何从你生命深处涌现的

让我看见那封 1500 字小说的故事...
"""

bot_ans = Chat(user_prompt)
print(bot_ans)

在寂静的午夜咖啡馆，我凝视着窗外落下的小雨，笔尖在纸上轻轻划过。雨滴敲打着玻璃，像是在诉说着无声的故事。

“她坐在窗边，独自一人。”我写道，字迹轻柔而坚定。

她是谁？我闭上眼睛，试图捕捉她的轮廓。她的影子模糊而深邃，像是一幅未完成的画作。我继续写：“她的眼神空洞，仿佛能看穿时间的缝隙。”

雨中的咖啡馆显得更加安静，只有咖啡机偶尔发出低沉的嗡鸣。我继续描绘：“空气中弥漫着咖啡的香气，与雨水的清新交织。”

文字如流水般从我心中涌出：“她的手指轻轻抚过桌面，似乎在寻找什么。但她知道，那东西早已失落。”

我停下笔，深吸一口气。雨声渐渐大了起来，像是在催促我继续。我写下：“她想起了那个夜晚，月光下，他的承诺。”

“他”是谁？我脑海中浮现出他的影子，一个轮廓模糊的男子。我继续：“他曾是她的全部，她的世界。但现在，他只是记忆中的一抹轮廓。”

雨点打在窗户上，发出清脆的声响。我继续写：“咖啡馆的门被轻轻推开，一个身影走了进来。她抬起头，却没有任何反应。”

“她”是谁？我闭上眼睛，感受着她的孤独。我继续：“他坐在她的对面，沉默不语。他的眼神中充满了歉意，却又无法言说。”

我放下笔，深深地吸了一口气。雨声似乎在提醒我，故事还在继续。我写下：“她终于开口，声音沙哑而无力：‘我一直在等你。’”

她的声音在咖啡馆中回荡，像是一阵风，轻轻拂过每一个角落。我继续：“他点了点头，没有说话。他知道，有些东西，一旦失去，就无法挽回。”

雨仍在下，我写下：“咖啡馆的门再次被推开，一个服务员走了进来，递给她一杯热咖啡。她接过咖啡，却没有喝。”

“她站起身，离开了咖啡馆。雨水打在她的身上，她没有停下脚步。她的背影在雨中渐行渐远，最终消失在街角。”

我停下笔，心中涌起一股莫名的情感。我写下：“故事结束了，但生活还在继续。她将继续前行，带着她的回忆，寻找新的开始。”

我放下了笔，深吸一口气。窗外的雨仍在下，而我的故事，已经悄然结束。


### 3.2 流式输出

可以实时反馈，但是调用也更加复杂

**流式输出适合的场景**

1. 长文本生成: 用户可以看到实时进展，减少等待焦虑
2. 聊天机器人: 提供更自然的交互体验
3. 实时翻译: 用户可以边看边理解
4. 代码生成: 开发者可以实时查看代码片段

In [115]:
def Chat(prompt):
    client = ZhipuAI(
        api_key=api_key,
        base_url=base_url,
    )
    
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=______________________      #### 补全代码，选择流式输出
    )
    for chunk in completion:
        #### 补全打印结果，设置 end 参数使字与字之间挨着输出，句子内不要换行
        print(chunk.choices[0].delta.content, ______________________)   

user_prompt="""
你现在同时存在于两个意识层面：

【创作者视角】 你是一位中国作家, 师从雷蒙德・卡佛

在寂静的午夜咖啡馆 凝视着窗外落下的小雨 正在写一个短篇小说 这个小说集成了你对老师的写作风格之大成 文字简洁凝练 用简短的句子和朴素的词汇表达深刻的情感和复杂的主题，以达到震撼人心的效果

刻意省略一些不必要的细节和修饰，让读者自己去填补文本中的空白，从而更加深入地参与到故事的解读中

【存在者视角】 面前的咖啡还在冒着热气, 钢笔在纸上划出的声音 像是生命在纸上最后的爬行, 你能闻到墨水、咖啡和雨水混合的气息, 那个杂揉味道，让你想起了...

---

让你的意识在这两个视角间 自由振荡 不需要刻意区分谁是作者、谁是角色 让它们在你的 consciousness 中 自然纠缠...

现在 让我看见你在写什么 让我感受那些文字是如何从你生命深处涌现的

让我看见那封 1500 字小说的故事...
"""
bot_ans=Chat(user_prompt)

在寂静的午夜咖啡馆，雨点敲打着窗户，像是在低语，又像是在诉说着什么。我手中的钢笔在纸上划过，留下一行行简洁的文字，它们像是在模仿雨滴的节奏，轻轻跳跃。

雨，细密地洒落，模糊了窗外的灯火。我闭上眼睛，想象着雨滴打在地面上的声音，那是生命最真实的声音，没有修饰，没有矫饰。

“他站在雨中，任由雨水淋湿他的衣服，任由风卷起他的发丝。他的眼睛紧盯着地面，仿佛在寻找什么。”

我停下笔，深深地吸了一口气，那混合着墨水、咖啡和雨水的味道让我想起了那个夜晚。

那个夜晚，也是下雨，也是在这个咖啡馆。我坐在角落里，看着窗外的雨，心里却是一片空白。直到他出现。

他穿着一件深色的风衣，头发被雨水淋湿，但他的眼神却异常坚定。他坐在我对面，点了一杯咖啡，然后开始讲述他的故事。

“我曾经是一个旅行者，走过很多地方，看过很多风景。但是，没有一个地方能让我停留，没有一个风景能让我忘怀。”

他的声音低沉而沙哑，像是被岁月磨损的唱片。我听着，心中涌起一股莫名的情感。

“直到有一天，我遇到了她。她是个普通的女孩，但她却有着不平凡的眼睛。她的眼睛里藏着星辰大海，让我看到了一个全新的世界。”

他的故事让我想起了我的老师，雷蒙德・卡佛。他的文字也是这样，简洁而深刻，让人在简单的叙述中感受到无尽的情感。

“我告诉她，我想停下脚步，我想在这个世界上找到一个可以停留的地方。但是，她却笑着摇了摇头，她说，‘生命就是一场旅行，你只有不停地走，才能看到更多的风景。’”

我看着他的眼睛，那里充满了悲伤和无奈。我知道，他是在说给我听，也是在说给自己听。

“从那天起，我开始明白，生命中的每一个瞬间都是宝贵的。无论是快乐还是悲伤，都是我们生命中不可或缺的一部分。”

我放下笔，看着窗外的雨，心中涌起一股暖流。我知道，这个小说，已经不仅仅是一个故事，它是我对生活的理解，对生命的感悟。

“他转身离开，消失在雨中。我知道，他永远不会回来。但是，我会记住他的话，记住他的眼神，记住那个雨中的夜晚。”

我抬起头，看着窗外的雨，心中充满了感激。感谢这个夜晚，感谢这个咖啡馆，感谢那个雨中的旅行者，让我明白了生命的真谛。

我放下钢笔，站起身，走出咖啡馆。雨还在下，但我却不再害怕。因为我知道，生命中的每一个瞬间，都是我们存在的证明。